In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/simuteh-2330511058

import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.praproses import praproses, simpan_param
from src.utils_cv import kontras_bercak

os.makedirs("hasil", exist_ok=True)

Mounted at /content/drive
/content/drive/MyDrive/simuteh-2330511058


In [ ]:
from src.praproses import praproses, simpan_param
import pandas as pd
import cv2

from src.utils_cv import baca, kontras_bercak

# Parameter praproses
param = {
    "gamma": 1.0,
    "clahe_clip": 2.0,
    "penapis": "median",
    "kernel": 3
}

simpan_param(param, "hasil/param_praproses.json")

# Kanal terbaik dari Bagian 2
KANAL = 2  # 0 = B, 1 = G, 2 = R

label = pd.read_csv("data/sintetis/label.csv")
baris = []

for _, d in label[label.kelas != "sehat"].iterrows():

    img = baca(f"data/sintetis/citra/{d.nama_file}")

    md = cv2.imread(
        f"data/sintetis/mask_daun/{d.nama_file}", 0
    )

    mb = cv2.imread(
        f"data/sintetis/mask_bercak/{d.nama_file}", 0
    )

    out = praproses(img, param)

    baris.append([
        d.kondisi_cahaya,
        kontras_bercak(img[:, :, KANAL], md, mb),
        kontras_bercak(out[:, :, KANAL], md, mb)
    ])

evaluasi = pd.DataFrame(
    baris,
    columns=["kondisi", "KB_sebelum", "KB_sesudah"]
)

print(evaluasi.groupby("kondisi").mean().round(2))

                KB_sebelum  KB_sesudah
kondisi                               
bayangan              5.20        8.98
gelap                 8.79       14.96
kontras_rendah        3.89        4.98
normal                8.51       10.54
terang                7.15        6.45
